# Data Quality Check
**Polymarket Gold Scraper — Sanity Checks on S3 Snapshot**

This notebook:
1. Downloads the **latest daily backup** from S3 into `./Data/`
2. Runs a suite of SQL-based sanity checks against it

> Set your AWS credentials in the CONFIG cell before running.

In [17]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
import os

BUCKET_NAME         = "thesis-data-ab3rnhard"
S3_PREFIX           = "backups/"          # prefix used when uploading
LOCAL_DATA_DIR      = "./Data"            # where to save the downloaded DB
TABLE_NAME          = "markets"           # main table in the SQLite DB
SCRAPED_AT_COL      = "scraped_at"        # timestamp column name
MARKET_ID_COL       = "id"                # unique market identifier column in this DB

# Read AWS settings from environment variables (PowerShell/system env).
AWS_ACCESS_KEY_ID     = (os.getenv("AWS_ACCESS_KEY_ID") or "").strip()
AWS_SECRET_ACCESS_KEY = (os.getenv("AWS_SECRET_ACCESS_KEY") or "").strip()
AWS_REGION            = (os.getenv("AWS_REGION") or "eu-north-1").strip() or "eu-north-1"

In [18]:
# Install dependency if missing
try:
    import boto3  # noqa: F401
except ModuleNotFoundError:
    %pip install boto3 -q

In [19]:
# Close any leftover connection from a previous run before downloading
try:
    conn.close()
    print("Closed leftover DB connection.")
except:
    pass  # conn didn't exist yet, that's fine

import boto3
import os
from pathlib import Path
from botocore.exceptions import ClientError

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

missing_vars = []
if not AWS_ACCESS_KEY_ID:
    missing_vars.append("AWS_ACCESS_KEY_ID")
if not AWS_SECRET_ACCESS_KEY:
    missing_vars.append("AWS_SECRET_ACCESS_KEY")

if missing_vars:
    raise ValueError(
        "Missing AWS environment variable(s): "
        + ", ".join(missing_vars)
        + ". Set them in PowerShell and restart the notebook kernel before rerunning."
    )

s3 = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION
)

# List all backup objects and pick the most recent one
try:
    response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=S3_PREFIX)
except ClientError as e:
    error_code = e.response.get('Error', {}).get('Code', 'UnknownError')
    raise RuntimeError(
        f"AWS S3 request failed ({error_code}). Check AWS key/secret, region, and S3 bucket permissions."
    ) from e

objects  = response.get('Contents', [])

if not objects:
    raise FileNotFoundError(f"No objects found in s3://{BUCKET_NAME}/{S3_PREFIX}")

latest_obj = sorted(objects, key=lambda x: x['LastModified'], reverse=True)[0]
s3_key     = latest_obj['Key']
file_date  = latest_obj['LastModified'].strftime('%Y-%m-%d')
local_name = f"local_database_{file_date}.sqlite"
local_path = Path(LOCAL_DATA_DIR) / local_name

print(f"Latest S3 object : {s3_key}")
print(f"Last modified    : {latest_obj['LastModified']}")
print(f"Size             : {latest_obj['Size'] / 1024:.1f} KB")
print(f"Saving to        : {local_path}")

s3.download_file(BUCKET_NAME, s3_key, str(local_path))
print("\nDownload complete.")

Closed leftover DB connection.
Latest S3 object : backups/db_2026-04-01.sqlite
Last modified    : 2026-04-01 22:02:12+00:00
Size             : 393672.0 KB
Saving to        : Data\local_database_2026-04-01.sqlite

Download complete.


In [20]:
import sqlite3
import pandas as pd
from datetime import datetime, timedelta, timezone

conn   = sqlite3.connect(str(local_path))
cursor = conn.cursor()

def q(sql, label=None):
    """Run a SQL query and return a formatted DataFrame."""
    if label:
        print(f"\n{'─'*60}")
        print(f"  {label}")
        print(f"{'─'*60}")
    df = pd.read_sql_query(sql, conn)
    print(df.to_string(index=False))
    return df

print(f"Connected to: {local_path}")
print(f"Check time  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Connected to: Data\local_database_2026-04-01.sqlite
Check time  : 2026-04-02 16:53:19


In [21]:
q(f"""
    SELECT COUNT(*) AS total_rows
    FROM {TABLE_NAME}
""", label="3.1  Total row count")


────────────────────────────────────────────────────────────
  3.1  Total row count
────────────────────────────────────────────────────────────
 total_rows
      47927


,total_rows
0,47927


In [22]:
cutoff_24h = (datetime.now(timezone.utc) - timedelta(hours=24)).strftime('%Y-%m-%d %H:%M:%S')

q(f"""
    SELECT
        COUNT(*)                     AS rows_last_24h,
        MIN({SCRAPED_AT_COL})        AS earliest_in_window,
        MAX({SCRAPED_AT_COL})        AS latest_in_window
    FROM {TABLE_NAME}
    WHERE {SCRAPED_AT_COL} >= '{cutoff_24h}'
""", label=f"3.2  Rows added in the last 24 h  (since {cutoff_24h} UTC)")


────────────────────────────────────────────────────────────
  3.2  Rows added in the last 24 h  (since 2026-04-01 14:53:19 UTC)
────────────────────────────────────────────────────────────
 rows_last_24h      earliest_in_window        latest_in_window
          7220 2026-04-01 14:55:42.295 2026-04-01 22:00:39.137


,rows_last_24h,earliest_in_window,latest_in_window
0,7220,2026-04-01 14:55:42.295,2026-04-01 22:00:39.137


In [23]:
q(f"""
    SELECT
        MIN({SCRAPED_AT_COL}) AS first_observation,
        MAX({SCRAPED_AT_COL}) AS last_observation
    FROM {TABLE_NAME}
""", label="3.3  First and last observation")


────────────────────────────────────────────────────────────
  3.3  First and last observation
────────────────────────────────────────────────────────────
      first_observation        last_observation
2026-03-30 15:15:52.586 2026-04-01 22:00:39.137


,first_observation,last_observation
0,2026-03-30 15:15:52.586,2026-04-01 22:00:39.137


In [24]:
q(f"""
    SELECT
        DATE({SCRAPED_AT_COL})           AS date,
        COUNT(*)                         AS rows_that_day,
        COUNT(DISTINCT {SCRAPED_AT_COL}) AS distinct_timestamps
    FROM {TABLE_NAME}
    GROUP BY DATE({SCRAPED_AT_COL})
    ORDER BY date
""", label="3.4  Row count per day")


────────────────────────────────────────────────────────────
  3.4  Row count per day
────────────────────────────────────────────────────────────
      date  rows_that_day  distinct_timestamps
2026-03-30           1830                   15
2026-03-31          20103                  158
2026-04-01          25994                  262


,date,rows_that_day,distinct_timestamps
0,2026-03-30,1830,15
1,2026-03-31,20103,158
2,2026-04-01,25994,262


In [25]:
dupe_df = q(f"""
    SELECT
        {MARKET_ID_COL},
        {SCRAPED_AT_COL},
        COUNT(*) AS occurrences
    FROM {TABLE_NAME}
    GROUP BY {MARKET_ID_COL}, {SCRAPED_AT_COL}
    HAVING COUNT(*) > 1
    ORDER BY occurrences DESC
    LIMIT 20
""", label="3.5  Duplicate rows (market_id + scraped_at appearing > 1×) — top 20")

total_dupes_df = pd.read_sql_query(f"""
    SELECT COUNT(*) AS duplicate_pairs
    FROM (
        SELECT {MARKET_ID_COL}, {SCRAPED_AT_COL}
        FROM {TABLE_NAME}
        GROUP BY {MARKET_ID_COL}, {SCRAPED_AT_COL}
        HAVING COUNT(*) > 1
    )
""", conn)
n_dupes = total_dupes_df.iloc[0]['duplicate_pairs']
print(f"\n  → Total duplicate (market_id, scraped_at) pairs: {n_dupes}")
if n_dupes == 0:
    print("  ✅ No duplicates found.")
else:
    print("  ⚠️  Duplicates detected — investigate!")


────────────────────────────────────────────────────────────
  3.5  Duplicate rows (market_id + scraped_at appearing > 1×) — top 20
────────────────────────────────────────────────────────────
Empty DataFrame
Columns: [id, scraped_at, occurrences]
Index: []

  → Total duplicate (market_id, scraped_at) pairs: 0
  ✅ No duplicates found.


In [26]:
cursor.execute(f"PRAGMA table_info({TABLE_NAME})")
columns = [row[1] for row in cursor.fetchall()]

null_parts = ",\n    ".join(
    [f"SUM(CASE WHEN {col} IS NULL OR CAST({col} AS TEXT) = '' THEN 1 ELSE 0 END) AS [{col}_nulls]"
     for col in columns]
)

null_df = pd.read_sql_query(f"SELECT\n    {null_parts}\nFROM {TABLE_NAME}", conn)

total_rows = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {TABLE_NAME}", conn).iloc[0]['n']
summary = pd.DataFrame({
    'column':     columns,
    'null_count': [null_df.iloc[0][f'{col}_nulls'] for col in columns]
})
summary['null_pct'] = (summary['null_count'] / total_rows * 100).round(2)
summary['status']   = summary['null_count'].apply(lambda x: '✅' if x == 0 else '⚠️ ')

print(f"\n{'─'*60}")
print("  3.6  NULL / empty value count per column")
print(f"{'─'*60}")
print(summary.to_string(index=False))


────────────────────────────────────────────────────────────
  3.6  NULL / empty value count per column
────────────────────────────────────────────────────────────
                      column  null_count  null_pct status
                          id           0      0.00      ✅
                  scraped_at           0      0.00      ✅
             acceptingOrders           0      0.00      ✅
    acceptingOrdersTimestamp           0      0.00      ✅
                      active           0      0.00      ✅
                    approved           0      0.00      ✅
                    archived           0      0.00      ✅
         automaticallyActive           0      0.00      ✅
                     bestAsk           0      0.00      ✅
                     bestBid        7479     15.60    ⚠️ 
            clearBookOnStart           0      0.00      ✅
                 clobRewards       32902     68.65    ⚠️ 
                clobTokenIds           0      0.00      ✅
                      

In [27]:
gap_df = pd.read_sql_query(f"""
    SELECT
        {SCRAPED_AT_COL} AS ts,
        LAG({SCRAPED_AT_COL}) OVER (ORDER BY {SCRAPED_AT_COL}) AS prev_ts
    FROM (
        SELECT DISTINCT {SCRAPED_AT_COL} FROM {TABLE_NAME}
    )
""", conn)

gap_df['ts']      = pd.to_datetime(gap_df['ts'])
gap_df['prev_ts'] = pd.to_datetime(gap_df['prev_ts'])
gap_df['gap_min'] = (gap_df['ts'] - gap_df['prev_ts']).dt.total_seconds() / 60
top_gaps          = gap_df.dropna().nlargest(10, 'gap_min')[['prev_ts', 'ts', 'gap_min']]
top_gaps.columns  = ['gap_start', 'gap_end', 'gap_minutes']
top_gaps['gap_minutes'] = top_gaps['gap_minutes'].round(1)

print(f"\n{'─'*60}")
print("  3.7  Top-10 largest gaps between consecutive snapshots")
print(f"{'─'*60}")
print(top_gaps.to_string(index=False))

expected_interval_min = 5   # adjust if your scraper interval differs
large_gaps = top_gaps[top_gaps['gap_minutes'] > expected_interval_min * 3]
if large_gaps.empty:
    print(f"\n  ✅ No gaps larger than {expected_interval_min*3} min detected.")
else:
    print(f"\n  ⚠️  {len(large_gaps)} gap(s) longer than {expected_interval_min*3} min — possible scraper downtime.")


────────────────────────────────────────────────────────────
  3.7  Top-10 largest gaps between consecutive snapshots
────────────────────────────────────────────────────────────
              gap_start                 gap_end  gap_minutes
2026-03-30 16:35:26.386 2026-03-31 10:35:47.777       1080.4
2026-03-31 10:45:42.434 2026-03-31 11:00:43.264         15.0
2026-03-30 16:12:30.438 2026-03-30 16:25:34.286         13.1
2026-04-01 13:25:36.422 2026-04-01 13:35:51.267         10.2
2026-03-31 11:55:45.930 2026-03-31 12:05:48.247         10.0
2026-04-01 10:10:27.896 2026-04-01 10:20:27.624         10.0
2026-04-01 10:35:37.254 2026-04-01 10:45:32.803          9.9
2026-03-30 15:21:07.515 2026-03-30 15:30:41.816          9.6
2026-03-31 17:10:40.279 2026-03-31 17:16:20.237          5.7
2026-03-31 18:55:46.882 2026-03-31 19:01:26.097          5.7

  ⚠️  1 gap(s) longer than 15 min — possible scraper downtime.


In [28]:
coverage_df = pd.read_sql_query(f"""
    SELECT
        question,
        COUNT(DISTINCT {SCRAPED_AT_COL}) AS snapshots,
        MIN({SCRAPED_AT_COL})            AS first_seen,
        MAX({SCRAPED_AT_COL})            AS last_seen
    FROM {TABLE_NAME}
    GROUP BY question
    ORDER BY snapshots DESC
""", conn)

print(f"\n{'─'*60}")
print("  3.8a  Top-10 most-scraped markets")
print(f"{'─'*60}")
print(coverage_df.head(10).to_string(index=False))

print(f"\n{'─'*60}")
print("  3.8b  Bottom-10 least-scraped markets")
print(f"{'─'*60}")
print(coverage_df.tail(10).to_string(index=False))


────────────────────────────────────────────────────────────
  3.8a  Top-10 most-scraped markets
────────────────────────────────────────────────────────────
                                           question  snapshots              first_seen               last_seen
    Will Gold (GC) hit (LOW) $3,400 by end of June?        435 2026-03-30 15:15:52.586 2026-04-01 22:00:39.137
   Will Gold (GC) hit (HIGH) $8,000 by end of June?        435 2026-03-30 15:15:52.586 2026-04-01 22:00:39.137
   Will Gold (GC) hit (HIGH) $7,000 by end of June?        435 2026-03-30 15:15:52.586 2026-04-01 22:00:39.137
   Will Gold (GC) hit (HIGH) $5,500 by end of June?        435 2026-03-30 15:15:52.586 2026-04-01 22:00:39.137
              Will Bitcoin outperform Gold in 2026?        435 2026-03-30 15:15:52.586 2026-04-01 22:00:39.137
    Will Bitcoin have the best performance in 2026?        435 2026-03-30 15:15:52.586 2026-04-01 22:00:39.137
Will the S&P 500 have the best performance in 2026?        434 2

In [29]:
print("\n" + "="*60)
print("  DATA QUALITY SUMMARY")
print("="*60)

rows_24h  = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {TABLE_NAME} WHERE {SCRAPED_AT_COL} >= '{cutoff_24h}'", conn).iloc[0]['n']
first_obs = pd.read_sql_query(f"SELECT MIN({SCRAPED_AT_COL}) AS ts FROM {TABLE_NAME}", conn).iloc[0]['ts']
last_obs  = pd.read_sql_query(f"SELECT MAX({SCRAPED_AT_COL}) AS ts FROM {TABLE_NAME}", conn).iloc[0]['ts']
null_issues = (summary['null_count'] > 0).sum()
max_gap     = top_gaps['gap_minutes'].max() if not top_gaps.empty else 0

print(f"  Total rows         : {total_rows:,}")
print(f"  Rows last 24 h     : {rows_24h:,}")
print(f"  First observation  : {first_obs}")
print(f"  Last observation   : {last_obs}")
print(f"  Duplicate pairs    : {n_dupes}  {'✅' if n_dupes == 0 else '⚠️'}")
print(f"  Columns with NULLs : {null_issues}  {'✅' if null_issues == 0 else '⚠️'}")
print(f"  Largest gap (min)  : {max_gap:.1f}  {'✅' if max_gap <= expected_interval_min*3 else '⚠️'}")
print("="*60)

conn.close()
print("Connection closed.")


  DATA QUALITY SUMMARY
  Total rows         : 47,927
  Rows last 24 h     : 7,220
  First observation  : 2026-03-30 15:15:52.586
  Last observation   : 2026-04-01 22:00:39.137
  Duplicate pairs    : 0  ✅
  Columns with NULLs : 43  ⚠️
  Largest gap (min)  : 1080.4  ⚠️
Connection closed.
